# RG3 최근10샷 표준편차의 실제 측정수 감사

109번의NaN 여부와 구분: min_periods5라 값이 있어도 실제 측정은5개뿐일 수 있다. 각 history_segment 내 이전10개의 유효 센서값 수를 계산. 현재·미래값·라벨은 횟수 계산에서 미사용. 데이터 관측 시작은 설비 가동 시작이라는 증거가 아님. 모든620개/가스19개 평가 범위 유지.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/109_rg3_history_availability_audit.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split("a=d[")[0])
keys=['EQUIP_CD','product','TimeStamp'];counts=[]
for sensor in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']:
 flag=observed.groupby('history_segment')[sensor].shift(1).notna().astype(int)
 name=sensor+'__past10_available_count'
 observed[name]=flag.groupby(observed.history_segment).transform(lambda v:v.rolling(10,min_periods=1).sum()).fillna(0)
 counts.append(name)
a=d[keys].merge(observed[keys+counts],on=keys,validate='one_to_one')
a['actual_gas']=y;a['block']=groups.to_numpy()
a['min_std4_history_count']=a[counts].min(axis=1)
a['min_noise3_history_count']=a[[counts[0],counts[2],counts[3]]].min(axis=1)
a['full10_std4']=a.min_std4_history_count.eq(10)
a['full10_noise3']=a.min_noise3_history_count.eq(10)
a.to_csv(t/'rg3_full10_history_availability.csv',index=False,encoding='utf-8-sig')
summary=a.groupby(['actual_gas','full10_std4','full10_noise3']).size().reset_index(name='rows')
summary.to_csv(t/'rg3_full10_history_summary.csv',index=False,encoding='utf-8-sig')
print(summary.to_string(index=False))
print(a[(a.actual_gas==1)&(~a.full10_noise3)][['TimeStamp','min_noise3_history_count']].to_string(index=False))
print('All620 evaluation rows retained. History counts are sensor coverage, NOT proof of machine startup.')


 actual_gas  full10_std4  full10_noise3  rows
          0        False          False    20
          0         True           True   581
          1        False          False     1
          1         True           True    18
          TimeStamp  min_noise3_history_count
2020-10-23 01:03:33                       2.0
All620 evaluation rows retained. History counts are sensor coverage, NOT proof of machine startup.
